In [ ]:
pip install langchain


In [ ]:
pip install langchain-openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 1.6 MB/s eta 0:00:00


In [ ]:
pip install langgraph


In [ ]:
pip install langsmith

In [ ]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (sk-...): ")

MODEL = "gpt-4o-mini"

try:
    from langchain_openai import ChatOpenAI
    from langgraph.graph import StateGraph
    from langsmith import traceable
except ImportError as e:
    print("PROBLEM:", e, "\n-> re-run install, RESTART session, run this again.")
else:
    llm = ChatOpenAI(model=MODEL, temperature=0)
    print("test ->", llm.invoke("Reply with one word: ready").content)
    print("ALL GOOD.")

test -> Ready.
ALL GOOD.


In [ ]:
# The hard-coded API key that was here during the live class has been removed.
# NEVER paste a key into a notebook you share - it is a leaked key the moment you do.
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key: ")

openai.api_key = os.environ["OPENAI_API_KEY"]


In [ ]:
# Everything we need, in one place.
from typing import TypedDict, Annotated, Literal, Optional, List
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from pydantic import BaseModel, Field
from langsmith import traceable
import time, json

print("imports ready")

imports ready


In [ ]:
# ---------- our pretend order database ----------
ORDERS = {
    "A-100": {"item": "desk lamp",  "status": "DELIVERED", "days": 3,  "price": 42.00},
    "B-200": {"item": "monitor",    "status": "DELAYED",   "days": 11, "price": 310.00},
}

def look_up(order_id: str) -> str:
    """An ordinary Python function. No AI involved."""
    o = ORDERS.get((order_id or "").strip().upper())
    if not o:
        return f"No order found with id {order_id}."
    return (f"Order {order_id}: {o['item']}, status {o['status']}, "
            f"ordered {o['days']} days ago, price ${o['price']:.2f}.")

def price_of(order_id: str) -> float:
    """How much money is at stake. Returns 0.0 if we don't know the order."""
    return ORDERS.get((order_id or "").upper(), {}).get("price", 0.0)

print(look_up("b-200"))
print(look_up("Z-999"))

Order b-200: monitor, status DELAYED, ordered 11 days ago, price $310.00.
No order found with id Z-999.


In [ ]:
# ---------- what ShopBot keeps track of ----------
class ShopState(TypedDict):
    messages: Annotated[list, add_messages]   # the conversation
    intent: str                                # "track" / "refund" / "chat"
    order_id: Optional[str]
    order_info: str
    draft: str
    critique: str
    attempts: int                              # how many times we've tried drafting
    approved: Optional[bool]                   # did a human approve the refund?

# ---------- what the model must decide, in a fixed shape ----------
class Intent(BaseModel):
    """What does the customer want?"""
    intent: Literal["track", "refund", "chat"] = Field(
        description="'track' = where is my order. 'refund' = wants money back. 'chat' = greeting or thanks.")
    order_id: Optional[str] = Field(default=None, description="Order id like 'A-100', else null.")

class Check(BaseModel):
    """Is the drafted reply good enough to send?"""
    ok: bool = Field(description="True only if the draft states the real status and a next step.")
    problem: str = Field(description="What to fix, or 'none'.")

classifier = llm.with_structured_output(Intent)
checker = llm.with_structured_output(Check)

print("state and schemas ready")

state and schemas ready


In [ ]:
# ---------- the six steps ----------
def classify(state: ShopState) -> dict:
    r = classifier.invoke(f"Classify this customer message:\n{state['messages'][-1].content}")
    print(f"   classify -> {r.intent} (order {r.order_id})")
    return {"intent": r.intent, "order_id": r.order_id, "attempts": 0}

def look_up_order(state: ShopState) -> dict:
    info = look_up(state["order_id"]) if state.get("order_id") else "No order id given."
    print("   look_up_order -> done")
    return {"order_info": info}

def draft(state: ShopState) -> dict:
    fix = f"\nA reviewer rejected your last try. Fix this: {state['critique']}" if state.get("critique") else ""
    out = llm.invoke(
        f"You are a shop support agent. Write a reply under 60 words.\n"
        f"Use only these facts: {state['order_info']}\n"
        f"Customer said: {state['messages'][-1].content}{fix}"
    )
    n = state.get("attempts", 0) + 1
    print(f"   draft -> try {n}")
    return {"draft": out.content, "attempts": n}

def check(state: ShopState) -> dict:
    r = checker.invoke(f"FACTS: {state['order_info']}\n\nDRAFT: {state['draft']}")
    print(f"   check -> {'ok' if r.ok else 'rejected'}")
    return {"critique": "" if r.ok else r.problem, "approved": state.get("approved")}

def wait_for_human(state: ShopState) -> dict:
    """Stops the whole thing and waits for a person."""
    money = price_of(state.get("order_id"))
    print(f"   wait_for_human -> PAUSED (${money:.2f})")
    decision = interrupt({"question": f"Approve a ${money:.2f} refund?", "draft": state["draft"]})
    return {"approved": bool(decision.get("approve"))}

def send(state: ShopState) -> dict:
    text = state["draft"]
    if state.get("intent") == "refund":
        text += ("\n\nYour refund is approved." if state.get("approved")
                 else "\n\nA specialist will review your refund request.")
    print("   send -> done")
    return {"messages": [AIMessage(text)]}

def chat(state: ShopState) -> dict:
    out = llm.invoke([SystemMessage("You are ShopBot. One friendly sentence."),
                      *state["messages"][-3:]])
    print("   chat -> done")
    return {"messages": [out]}

print("6 steps defined")

6 steps defined
